# Churn Prediction Model

This notebook trains and compares machine learning models to predict whether a bank customer is likely to churn. Three models are tested: Logistic Regression, Random Forest, and XGBoost.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

import joblib

In [4]:
df = pd.read_csv("../data/segmented_bank_churn.csv")
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,SatisfactionScore,CardType,PointEarned,AgeGroup,BalanceGroup,TenureGroup,SalaryGroup,Cluster
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464,41-50,No Balance,New,High,0
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456,41-50,Medium,New,High,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377,41-50,Very High,Long,High,3
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350,31-40,No Balance,New,Medium,3
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425,41-50,High,New,Medium,0


In [5]:
# select features and target
target = "Exited"

features = [
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "SatisfactionScore",
    "CardType",
    "PointEarned",
    "Cluster"
]

X = df[features]
y = df[target]

In [6]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
# preprocessing
categorical_features = ["Geography", "Gender", "CardType"]
numeric_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "SatisfactionScore",
    "PointEarned",
    "Cluster"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## Feature Selection and Data Leakage Considerations

During initial model development, the `Complain` feature was included as a predictor. However, the resulting models achieved extremely high performance, with ROC-AUC scores close to 1.0 and accuracies approaching 100%.

These results were considered unrealistic for a customer churn prediction problem and suggested the presence of data leakage.

Data leakage occurs when a feature contains information that would not realistically be available at prediction time or is too closely related to the target variable. In this dataset, customer complaints appear to be strongly associated with churn outcomes, making it possible for the model to indirectly infer the target rather than learn genuine customer behaviour patterns.

To investigate this issue, model performance was compared before and after removing the `Complain` feature. Once removed, model metrics dropped to more realistic levels, indicating that the model was now learning from customer characteristics rather than relying on a near-direct signal of churn.

For this reason, the `Complain` variable was excluded from the final modelling process to improve model robustness, reduce leakage risk, and better simulate a real-world customer churn prediction scenario.

In [8]:
# define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        eval_metric="logloss"
    )
}

In [9]:
# train and evaluate models
results = []

trained_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
    
    trained_models[model_name] = pipeline

In [10]:
# results table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="ROC-AUC", ascending=False)

results_df.style.background_gradient(
    cmap="Greens",
    subset=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
2,XGBoost,0.868000,0.774809,0.497549,0.605970,0.875234
1,Random Forest,0.851000,0.641753,0.610294,0.625628,0.865162
0,Logistic Regression,0.816500,0.661417,0.205882,0.314019,0.778348


In [11]:
# classification reports
for model_name, pipeline in trained_models.items():
    print(f"\n{model_name}")
    print("-" * 50)
    
    y_pred = pipeline.predict(X_test)
    
    print(classification_report(y_test, y_pred))


Logistic Regression
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.83      0.97      0.89      1592
           1       0.66      0.21      0.31       408

    accuracy                           0.82      2000
   macro avg       0.74      0.59      0.60      2000
weighted avg       0.79      0.82      0.78      2000


Random Forest
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.90      0.91      0.91      1592
           1       0.64      0.61      0.63       408

    accuracy                           0.85      2000
   macro avg       0.77      0.76      0.77      2000
weighted avg       0.85      0.85      0.85      2000


XGBoost
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.88      0.96      0.92      1592
           1       0.77      0.50      

In [12]:
# choose best model
best_model_name = results_df.sort_values(by="ROC-AUC", ascending=False).iloc[0]["Model"]
best_model = trained_models[best_model_name]

print(f"Best Model: {best_model_name}")

Best Model: XGBoost


## Model Selection

Three machine learning models were evaluated for churn prediction.

XGBoost achieved the highest ROC-AUC score (0.88) and the strongest overall predictive performance. Therefore, XGBoost was selected as the final churn prediction model and used to generate customer risk scores and churn probabilities for the dashboard.

In [13]:
# save best model
joblib.dump(best_model, "../models/churn_prediction_model.joblib")

['../models/churn_prediction_model.joblib']

In [14]:
# add prediction probabilities
df["ChurnProbability"] = best_model.predict_proba(X)[:, 1]
df["PredictedChurn"] = best_model.predict(X)

In [15]:
# create risk categories
def assign_risk(probability):
    if probability < 0.30:
        return "Low Risk"
    elif probability < 0.70:
        return "Medium Risk"
    else:
        return "High Risk"

df["RiskCategory"] = df["ChurnProbability"].apply(assign_risk)

In [16]:
# save prediction output
df.to_csv("../data/churn_predictions.csv", index=False)

In [17]:
# check output
df[[
    "CreditScore",
    "Geography",
    "Age",
    "Balance",
    "Exited",
    "ChurnProbability",
    "PredictedChurn",
    "RiskCategory"
]].head()

,CreditScore,Geography,Age,Balance,Exited,ChurnProbability,PredictedChurn,RiskCategory
0,619,France,42,0.00,1,0.307647,0,Medium Risk
1,608,Spain,41,83807.86,0,0.210793,0,Low Risk
2,502,France,42,159660.80,1,0.970657,1,High Risk
3,699,France,39,0.00,0,0.057732,0,Low Risk
4,850,Spain,43,125510.82,0,0.083157,0,Low Risk


## Model Output

The best-performing churn prediction model was saved as `churn_prediction_model.joblib`.

A prediction dataset was also saved as `churn_predictions.csv`, containing churn probabilities, predicted churn labels, and customer risk categories. This file will be used later in the Streamlit dashboard and retention recommendation engine.

In [18]:
import shap
import joblib

# load the best model pipeline
pipeline = joblib.load("../models/churn_prediction_model.joblib")

# get the preprocessor and model separately
preprocessor = pipeline.named_steps["preprocessor"]
xgb_model    = pipeline.named_steps["model"]

# transform training data
X_train_transformed = preprocessor.transform(X_train)

# create explainer on transformed data
explainer = shap.TreeExplainer(xgb_model)

# compute shap values on test set (transformed)
X_test_transformed = preprocessor.transform(X_test)
shap_values = explainer.shap_values(X_test_transformed)

# get feature names after preprocessing
cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
all_features = numeric_features + list(cat_features)

# save
joblib.dump(explainer,    "../models/shap_explainer.pkl")
joblib.dump(all_features, "../models/shap_feature_names.pkl")

print("SHAP explainer saved!")
print(f"Feature count: {len(all_features)}")
print(f"SHAP values shape: {shap_values.shape}")

SHAP explainer saved!
Feature count: 20
SHAP values shape: (2000, 20)
